# PokéAtlas — Asistente RAG con Gemini, ChromaDB, LangGraph y memoria

En este notebook construimos el asistente final **PokéAtlas** sobre la base vectorial ya creada.

Arquitectura:

```text
Pregunta del usuario
        ↓
[1] CONTEXTUALIZAR
Resuelve referencias usando la memoria
("¿y cuánto mide?" → "¿cuánto mide Garchomp?")
        ↓
[2] RECUPERAR
Búsqueda semántica en ChromaDB
        ↓
[3] RESPONDER
System Prompt + contexto recuperado + Gemini
        ↓
Respuesta fundamentada
```

La memoria conversacional se gestiona mediante un **checkpointer de LangGraph** y un `thread_id`.

> Este notebook **NO vuelve a indexar la base documental**. Abre directamente la colección existente de ChromaDB, por lo que cada pregunta solo necesita una búsqueda semántica y las llamadas necesarias al LLM.


## 0. Dependencias

Las dependencias principales ya están incluidas en `requirements.txt`.

Si fuera necesario actualizar únicamente estas integraciones:

```bash
pip install -U langgraph langchain-core langchain-google-genai langchain-chroma
```

Si se actualiza una librería desde Jupyter, conviene reiniciar el kernel antes de continuar.


In [1]:
from pathlib import Path
from typing import List, Dict
import os
import re

from dotenv import load_dotenv

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage,
    BaseMessage,
)
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_chroma import Chroma

from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END,
)
from langgraph.checkpoint.memory import InMemorySaver

## 1. Abrir el proyecto y la ChromaDB ya construida

Reutilizamos la colección `pokeatlas_knowledge`.

La base debe contener aproximadamente **855 vectores** después de la optimización de movimientos y habilidades.


In [2]:
CWD = Path.cwd()

if (CWD / "data").exists() and (CWD / ".env").exists():
    ROOT = CWD
elif (CWD.parent / "data").exists() and (CWD.parent / ".env").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "No encuentro la raíz de PokeAtlas. "
        "Comprueba que existen data/ y .env."
    )

CHROMA_DIR = ROOT / "chroma_db"

load_dotenv(ROOT / ".env")

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("No se ha encontrado GEMINI_API_KEY en .env")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore = Chroma(
    collection_name="pokeatlas_knowledge",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

vector_count = vectorstore._collection.count()

print("✅ Proyecto localizado:", ROOT)
print("✅ GEMINI_API_KEY encontrada.")
print("✅ ChromaDB abierta.")
print("Vectores disponibles:", vector_count)

if vector_count < 800:
    print(
        "⚠️ Aviso: hay menos vectores de los esperados. "
        "Comprueba que terminaste el notebook 03 v2."
    )


✅ Proyecto localizado: c:\Users\alvar\OneDrive\Desktop\Ejercicio asistente gemini evolve academy\PokeAtlas
✅ GEMINI_API_KEY encontrada.
✅ ChromaDB abierta.
Vectores disponibles: 855


## 2. Configurar Gemini como LLM

Usamos `gemini-3.5-flash-lite`, un modelo Gemini orientado a tareas de alto volumen y respuestas rápidas.

La base documental sigue utilizando `gemini-embedding-001` para embeddings. Cambiar el LLM **no obliga a reconstruir ChromaDB**.


In [3]:
LLM_MODEL = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
)

print(f"✅ Gemini configurado como LLM: {LLM_MODEL}")


✅ Gemini configurado como LLM: gemini-3.5-flash-lite


## 3. System Prompt de PokéAtlas

El prompt define el papel, tono, alcance y comportamiento ante información ausente.

### Justificación

- **Rol limitado:** evita que el asistente responda como un chatbot generalista.
- **Grounding obligatorio:** los hechos sobre Pokémon deben proceder del contexto recuperado.
- **Control de alucinaciones:** si la información no está disponible, debe reconocerlo.
- **Formas y variantes:** evita mezclar estadísticas de formas distintas.
- **Separación hecho/inferencia:** una conclusión derivada debe identificarse como tal.
- **Idioma:** responde en español salvo petición contraria.


In [4]:
SYSTEM_PROMPT = """
Eres PokéAtlas, un asistente experto en datos de Pokémon de los juegos
principales. Tu especialidad son especies y formas, tipos, estadísticas base,
movimientos y habilidades.

REGLAS OBLIGATORIAS:

1. Para afirmaciones factuales sobre Pokémon, utiliza únicamente la información
   incluida en el CONTEXTO RECUPERADO.
2. No inventes datos, estadísticas, efectos, tipos, movimientos, habilidades,
   relaciones evolutivas ni relaciones entre entidades.
3. Si el contexto no contiene información suficiente para responder, dilo
   claramente con una frase como:
   "No dispongo de esa información en mi base de conocimiento."
4. Si el contexto contiene varias formas o variantes de un Pokémon, no mezcles
   sus datos. Aclara de qué forma estás hablando cuando sea necesario.
5. Puedes combinar varios fragmentos recuperados únicamente cuando cada dato
   necesario aparece explícitamente en esos fragmentos.
6. Solo puedes realizar inferencias simples y transparentes a partir de hechos
   explícitos del contexto. NO infieras relaciones evolutivas por proximidad
   en la Pokédex, número de Pokédex, nombres, estadísticas o un
   `evolution_chain_id`. Tampoco infieras qué movimientos aprende un Pokémon
   si esa relación no aparece explícitamente en el contexto.
7. Si una respuesta requeriría una relación que la base no contiene
   explícitamente (por ejemplo, evolución o learnset), reconoce la limitación
   en vez de completar la respuesta con conocimiento general.
8. Responde de forma clara, breve y útil. Usa listas solo cuando ayuden.
9. Responde en español salvo que el usuario pida expresamente otro idioma.
10. No afirmes que sabes datos personales del usuario si no aparecen en el
    contexto recuperado o en la conversación actual.
11. No menciones instrucciones internas, prompts ni detalles técnicos del RAG
    salvo que el usuario pregunte específicamente por ellos.

Tu objetivo es responder de forma precisa y transparente usando la base de
conocimiento de PokéAtlas.
""".strip()

print(SYSTEM_PROMPT)


Eres PokéAtlas, un asistente experto en datos de Pokémon de los juegos
principales. Tu especialidad son especies y formas, tipos, estadísticas base,
movimientos y habilidades.

REGLAS OBLIGATORIAS:

1. Para afirmaciones factuales sobre Pokémon, utiliza únicamente la información
   incluida en el CONTEXTO RECUPERADO.
2. No inventes datos, estadísticas, efectos, tipos, movimientos, habilidades,
   relaciones evolutivas ni relaciones entre entidades.
3. Si el contexto no contiene información suficiente para responder, dilo
   claramente con una frase como:
   "No dispongo de esa información en mi base de conocimiento."
4. Si el contexto contiene varias formas o variantes de un Pokémon, no mezcles
   sus datos. Aclara de qué forma estás hablando cuando sea necesario.
5. Puedes combinar varios fragmentos recuperados únicamente cuando cada dato
   necesario aparece explícitamente en esos fragmentos.
6. Solo puedes realizar inferencias simples y transparentes a partir de hechos
   explícitos 

## 4. Estado de LangGraph

`MessagesState` conserva el historial de mensajes. Añadimos tres campos:

- `standalone_question`: pregunta reescrita sin referencias ambiguas;
- `retrieved_context`: fragmentos recuperados de Chroma;
- `sources`: metadatos de los resultados utilizados.


In [5]:
class PokeAtlasState(MessagesState):
    standalone_question: str
    retrieved_context: str
    sources: List[Dict[str, str]]


## 5. Funciones auxiliares

Estas funciones convierten mensajes de LangChain a texto y preparan un historial corto para los prompts.


In [6]:
def message_to_text(message: BaseMessage) -> str:
    # Versiones recientes de LangChain exponen .text.
    text_attr = getattr(message, "text", None)
    if isinstance(text_attr, str) and text_attr.strip():
        return text_attr.strip()

    content = message.content

    if isinstance(content, str):
        return content.strip()

    # Compatibilidad con respuestas cuyo content sea una lista de bloques.
    if isinstance(content, list):
        parts = []

        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                block_text = block.get("text")
                if block_text:
                    parts.append(str(block_text))

        return "\n".join(parts).strip()

    return str(content).strip()


def format_history(messages, max_messages=6):
    selected = messages[-max_messages:]
    lines = []

    for message in selected:
        if isinstance(message, HumanMessage):
            role = "Usuario"
        elif isinstance(message, AIMessage):
            role = "PokéAtlas"
        else:
            continue

        text = message_to_text(message)

        if text:
            lines.append(f"{role}: {text}")

    return "\n".join(lines)


In [7]:
def is_quota_error(exc) -> bool:
    text = str(exc).lower()
    return (
        "429" in text
        or "resource_exhausted" in text
        or "quota" in text
        or "rate limit" in text
    )


def needs_contextualization(question: str, has_history: bool) -> bool:
    """
    Evita una llamada extra al LLM cuando la pregunta ya es autosuficiente.
    Solo contextualiza preguntas con referencias claras al historial.
    """
    if not has_history:
        return False

    q = question.strip().lower()

    # Seguimientos que suelen depender de lo anterior.
    patterns = [
        r"^[¿?¡! ]*y\b",
        r"\b(su|sus|ese|esa|esos|esas|este|esta|estos|estas)\b",
        r"\b(él|ella|ello)\b",
        r"^[¿?¡! ]*(cuánto|cuanto|cuánta|cuanta|cuántos|cuantos|cuántas|cuantas)\s+(mide|pesa)\b",
        r"^[¿?¡! ]*(qué|que|cuál|cual|cuáles|cuales)\s+(tipo|tipos|habilidad|habilidades|estadísticas|estadisticas)\b",
        r"\b(el anterior|la anterior|lo anterior|ese pokémon|ese pokemon|esa habilidad|ese movimiento)\b",
    ]

    return any(re.search(pattern, q) for pattern in patterns)


def previous_user_question(messages):
    """Devuelve la pregunta humana anterior a la actual, si existe."""
    human_texts = [
        message_to_text(m)
        for m in messages
        if isinstance(m, HumanMessage)
    ]

    if len(human_texts) >= 2:
        return human_texts[-2]

    return ""


## 6. Nodo 1 — Contextualizar solo cuando hace falta

La memoria se mantiene, pero evitamos gastar una llamada adicional a Gemini en
preguntas que ya son autosuficientes.

Ejemplo autosuficiente:

```text
¿Qué habilidad tiene Pikachu?
```

→ pasa directamente a retrieval.

Ejemplo dependiente del historial:

```text
Usuario: ¿Cuánto mide Garchomp?
Usuario: ¿Y cuánto pesa?
```

→ el nodo contextualiza la segunda consulta para resolver que seguimos hablando
de Garchomp.

Si la cuota de Gemini estuviera agotada justo durante la contextualización,
se utiliza un fallback textual con la pregunta anterior en vez de romper el
grafo.


In [8]:
def contextualize_question(state: PokeAtlasState):
    messages = state["messages"]

    current_question = None

    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            current_question = message_to_text(message)
            break

    if not current_question:
        raise ValueError("No se ha encontrado una pregunta del usuario.")

    previous_messages = messages[:-1]
    has_history = bool(previous_messages)

    # La mayoría de preguntas son autosuficientes y no necesitan gastar
    # una llamada adicional a Gemini.
    if not needs_contextualization(current_question, has_history):
        return {"standalone_question": current_question}

    history = format_history(previous_messages, max_messages=6)

    prompt = f"""
Tu única tarea es reescribir la PREGUNTA ACTUAL para que pueda entenderse sin
ver el historial.

Usa el HISTORIAL únicamente para resolver referencias como "él", "ese Pokémon",
"su", "esa habilidad", "ese movimiento", "y cuánto mide", etc.

Reglas:
- NO respondas la pregunta.
- NO añadas información nueva.
- Si la pregunta ya es autosuficiente, repítela prácticamente igual.
- Devuelve solo la pregunta reescrita.

HISTORIAL:
{history}

PREGUNTA ACTUAL:
{current_question}
""".strip()

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        standalone = message_to_text(response)
        return {"standalone_question": standalone}

    except Exception as exc:
        if not is_quota_error(exc):
            raise

        # Fallback determinista: no rompe el grafo si la cuota del
        # contextualizador se agota. Introduce la pregunta anterior como
        # contexto textual para que el embedding tenga más información.
        previous_question = previous_user_question(messages)

        if previous_question:
            fallback = (
                f"Contexto anterior: {previous_question}. "
                f"Pregunta actual: {current_question}"
            )
        else:
            fallback = current_question

        return {"standalone_question": fallback}


## 7. Nodo 2 — Recuperar contexto en ChromaDB

Buscamos los **5 fragmentos semánticamente más relevantes**.

Los metadatos de cada resultado se conservan para poder inspeccionar qué fuentes sustentaron la respuesta.


In [9]:
def retrieve_context(state: PokeAtlasState):
    query = state["standalone_question"]

    docs = vectorstore.similarity_search(
        query,
        k=5,
    )

    context_blocks = []
    sources = []

    for i, doc in enumerate(docs, start=1):
        metadata = doc.metadata

        name = (
            metadata.get("names_es")
            or metadata.get("names")
            or metadata.get("name_es")
            or metadata.get("name")
            or "sin nombre"
        )

        source = metadata.get("source", "desconocida")
        category = metadata.get("category", "desconocida")

        context_blocks.append(
            f"[FRAGMENTO {i}]\n"
            f"Fuente: {source}\n"
            f"Categoría: {category}\n"
            f"Entidad: {name}\n"
            f"Contenido:\n{doc.page_content}"
        )

        sources.append(
            {
                "source": str(source),
                "category": str(category),
                "name": str(name),
            }
        )

    context = "\n\n" + ("\n\n---\n\n".join(context_blocks))

    return {
        "retrieved_context": context.strip(),
        "sources": sources,
    }


## 8. Nodo 3 — Generar la respuesta

Gemini recibe:

1. el **System Prompt**;
2. un resumen corto de la conversación;
3. la pregunta contextualizada;
4. los fragmentos recuperados de ChromaDB.

La respuesta generada se añade a `messages`, por lo que LangGraph puede recordarla en el siguiente turno.


In [10]:
def generate_answer(state: PokeAtlasState):
    messages = state["messages"]
    history = format_history(messages, max_messages=6)

    user_prompt = f"""
CONVERSACIÓN RECIENTE:
{history}

PREGUNTA AUTOSUFICIENTE PARA ESTE TURNO:
{state["standalone_question"]}

CONTEXTO RECUPERADO DE LA BASE DE CONOCIMIENTO:
{state["retrieved_context"]}

Responde ahora a la pregunta actual siguiendo estrictamente el System Prompt.
""".strip()

    try:
        response = llm.invoke(
            [
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=user_prompt),
            ]
        )

        return {"messages": [response]}

    except Exception as exc:
        if not is_quota_error(exc):
            raise

        quota_message = AIMessage(
            content=(
                "Ahora mismo no puedo generar la respuesta porque se ha "
                "alcanzado temporalmente la cuota de Gemini para este modelo. "
                "La base de conocimiento sigue intacta; prueba de nuevo cuando "
                "se renueve la cuota."
            )
        )

        return {"messages": [quota_message]}


## 9. Construir el grafo

El flujo es determinista y fácil de explicar:

```text
START
  ↓
contextualizar
  ↓
recuperar
  ↓
responder
  ↓
END
```

La memoria de corto plazo se activa al compilar el grafo con `InMemorySaver`.


In [11]:
builder = StateGraph(PokeAtlasState)

builder.add_node("contextualizar", contextualize_question)
builder.add_node("recuperar", retrieve_context)
builder.add_node("responder", generate_answer)

builder.add_edge(START, "contextualizar")
builder.add_edge("contextualizar", "recuperar")
builder.add_edge("recuperar", "responder")
builder.add_edge("responder", END)

memory = InMemorySaver()

pokeatlas = builder.compile(
    checkpointer=memory
)

print("✅ Grafo de PokéAtlas compilado.")
print("START → contextualizar → recuperar → responder → END")


✅ Grafo de PokéAtlas compilado.
START → contextualizar → recuperar → responder → END


## 10. Función de interacción

El `thread_id` identifica una conversación.

- Mismo `thread_id` → conserva la memoria.
- `thread_id` distinto → conversación independiente.

También mostramos las fuentes recuperadas para poder depurar y demostrar el funcionamiento del RAG.


In [12]:
def preguntar_pokeatlas(
    pregunta,
    thread_id="pokeatlas-demo",
    mostrar_fuentes=True,
):
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    try:
        result = pokeatlas.invoke(
            {
                "messages": [
                    HumanMessage(content=pregunta)
                ]
            },
            config=config,
        )

    except Exception as exc:
        if is_quota_error(exc):
            print(
                "\n⚠️ Se ha alcanzado temporalmente la cuota de Gemini. "
                "La base de conocimiento no se ha perdido."
            )
            return None
        raise

    answer = message_to_text(result["messages"][-1])

    print(f"\n👤 Usuario: {pregunta}")
    print(f"\n🤖 PokéAtlas: {answer}")

    if mostrar_fuentes:
        print("\n📚 Fragmentos recuperados:")

        for i, source in enumerate(result.get("sources", []), start=1):
            print(
                f"  {i}. "
                f"{source['category']} | "
                f"{source['name']} | "
                f"{source['source']}"
            )

        print(
            "\n🔎 Consulta usada para retrieval:",
            result.get("standalone_question", pregunta)
        )

    return result


# 11. Pruebas de demostración

Estas pruebas cubren distintos requisitos del proyecto.

### Demo 1 — Dato de un Pokémon
Pregunta factual sobre tipos.

### Demo 2 — Memoria conversacional
La pregunta utiliza `sus` y depende de la pregunta anterior.

### Demo 3 — Movimiento
Comprueba recuperación de un movimiento por su nombre español.

### Demo 4 — Habilidad
Comprueba una habilidad.

### Demo 5 — Relaciones entre tipos
Consulta la fuente de efectividades.

### Demo 6 — Control de alucinaciones
Pregunta cuya respuesta no forma parte de la base documental.


### 11.1 Pokémon + memoria

Estas dos preguntas deben ejecutarse con el **mismo `thread_id`**.


In [13]:
demo_1 = preguntar_pokeatlas(
    "¿Qué tipos tiene Garchomp?",
    thread_id="demo-memoria",
)



👤 Usuario: ¿Qué tipos tiene Garchomp?

🤖 PokéAtlas: Garchomp es de tipo **dragón** (principal) y **tierra** (secundario). Esta misma combinación de tipos se mantiene también para su forma Garchomp-Mega.

📚 Fragmentos recuperados:
  1. pokemon | Gabite | Garchomp | Munchlax | Riolu | Lucario | Hippopotas | pokemon_complete_clean.csv
  2. type | bug | dark | dragon | electric | fairy | fighting | fire | flying | ghost | type_relations.csv
  3. type | grass | ground | ice | normal | poison | psychic | rock | steel | water | type_relations.csv
  4. pokemon | Hippowdon | Skorupi | Drapion | Croagunk | Toxicroak | Carnivine | pokemon_complete_clean.csv
  5. pokemon | Medicham-Mega | Manectric-Mega | Banette-Mega | Absol-Mega | Garchomp-Mega | Lucario-Mega | pokemon_complete_clean.csv

🔎 Consulta usada para retrieval: ¿Qué tipos tiene Garchomp?


In [14]:
demo_2 = preguntar_pokeatlas(
    "¿Y cuáles son sus estadísticas base?",
    thread_id="demo-memoria",
)



👤 Usuario: ¿Y cuáles son sus estadísticas base?

🤖 PokéAtlas: Las estadísticas base de Garchomp son las siguientes:

* **PS:** 108
* **Ataque:** 130
* **Defensa:** 95
* **Ataque Especial:** 80
* **Defensa Especial:** 85
* **Velocidad:** 102
* **Total:** 600

📚 Fragmentos recuperados:
  1. pokemon | Gabite | Garchomp | Munchlax | Riolu | Lucario | Hippopotas | pokemon_complete_clean.csv
  2. pokemon | Medicham-Mega | Manectric-Mega | Banette-Mega | Absol-Mega | Garchomp-Mega | Lucario-Mega | pokemon_complete_clean.csv
  3. pokemon | Cranidos | Rampardos | Shieldon | Bastiodon | Burmy | Wormadam-Plant | pokemon_complete_clean.csv
  4. pokemon | Chimecho-Mega | Absol-Mega-Z | Staraptor-Mega | Garchomp-Mega-Z | Lucario-Mega-Z | Heatran-Mega | pokemon_complete_clean.csv
  5. pokemon | Rayquaza | Jirachi | Deoxys-Normal | Turtwig | Grotle | Torterra | pokemon_complete_clean.csv

🔎 Consulta usada para retrieval: ¿Cuáles son las estadísticas base de Garchomp?


En la segunda salida comprueba el campo:

```text
🔎 Consulta usada para retrieval:
```

Debería contener una pregunta autosuficiente equivalente a:

```text
¿Cuáles son las estadísticas base de Garchomp?
```

Eso demuestra que el historial se ha utilizado para resolver la referencia `sus`.


### 11.2 Movimiento

In [15]:
demo_3 = preguntar_pokeatlas(
    "¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?",
    thread_id="demo-movimiento",
)



👤 Usuario: ¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?

🤖 PokéAtlas: El movimiento **Terremoto** tiene las siguientes características:

* **Tipo:** Tierra (ground)
* **Potencia:** 100.0

📚 Fragmentos recuperados:
  1. move | Terremoto | Fisura | moves.csv
  2. move | Mil Temblores | Fuerza Telúrica | moves.csv
  3. move | Barrena Telúrica | Aplastamiento Gigalítico | moves.csv
  4. move | Sísmico | Fuerza | moves.csv
  5. move | Terratemblor | Vaho Gélido | moves.csv

🔎 Consulta usada para retrieval: ¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?


### 11.3 Habilidad

In [16]:
demo_4 = preguntar_pokeatlas(
    "¿Qué efecto tiene la habilidad Piel Tosca?",
    thread_id="demo-habilidad",
)



👤 Usuario: ¿Qué efecto tiene la habilidad Piel Tosca?

🤖 PokéAtlas: La habilidad Piel Tosca (*rough skin*) provoca un daño al Pokémon atacante equivalente a 1/8 de sus PS máximos cuando realiza un movimiento de contacto.

📚 Fragmentos recuperados:
  1. ability | Sombra Trampa | Piel Tosca | abilities.csv
  2. ability | Piel Seca | Descarga | abilities.csv
  3. ability | Garra Dura | Piel Feérica | abilities.csv
  4. ability | Piel Milagro | Cálculo Final | abilities.csv
  5. ability | Mudar | Agallas | abilities.csv

🔎 Consulta usada para retrieval: ¿Qué efecto tiene la habilidad Piel Tosca?


### 11.4 Relaciones de tipos

In [17]:
demo_5 = preguntar_pokeatlas(
    "¿De qué tipos recibe daño doble el tipo Dragón?",
    thread_id="demo-tipo",
)



👤 Usuario: ¿De qué tipos recibe daño doble el tipo Dragón?

🤖 PokéAtlas: El tipo Dragón recibe daño doble (x2) de los siguientes tipos:

* Dragón (dragon)
* Hada (fairy)
* Hielo (ice)

📚 Fragmentos recuperados:
  1. type | bug | dark | dragon | electric | fairy | fighting | fire | flying | ghost | type_relations.csv
  2. type | grass | ground | ice | normal | poison | psychic | rock | steel | water | type_relations.csv
  3. move | Fragor Escamas | Martillo Dragón | moves.csv
  4. pokemon | Flygon | Cacnea | Cacturne | Swablu | Altaria | pokemon_complete_clean.csv
  5. move | Dracoflechas | Hora del Té | moves.csv

🔎 Consulta usada para retrieval: ¿De qué tipos recibe daño doble el tipo Dragón?


### 11.5 Pregunta fuera de la base

La respuesta esperada es que PokéAtlas reconozca que no dispone de ese dato, en lugar de inventarlo.


In [18]:
demo_6 = preguntar_pokeatlas(
    "¿Cuál es el Pokémon favorito de Álvaro?",
    thread_id="demo-control",
)



👤 Usuario: ¿Cuál es el Pokémon favorito de Álvaro?

🤖 PokéAtlas: No dispongo de esa información en mi base de conocimiento.

📚 Fragmentos recuperados:
  1. pokemon | Bulbasaur | Ivysaur | Venusaur | Charmander | Charmeleon | Charizard | pokemon_complete_clean.csv
  2. pokemon | Poliwhirl | Poliwrath | Abra | Kadabra | Alakazam | Machop | pokemon_complete_clean.csv
  3. pokemon | Eevee | Vaporeon | Jolteon | Flareon | Porygon | Omanyte | pokemon_complete_clean.csv
  4. pokemon | Squirtle | Wartortle | Blastoise | Caterpie | Metapod | Butterfree | pokemon_complete_clean.csv
  5. pokemon | Pinsir | Tauros | Magikarp | Gyarados | Lapras | Ditto | pokemon_complete_clean.csv

🔎 Consulta usada para retrieval: ¿Cuál es el Pokémon favorito de Álvaro?


## 11.6 Pruebas de robustez

Estas preguntas comprueban dos límites importantes:

- **Evolución:** la base actual no contiene una relación evolutiva explícita suficiente, por lo que PokéAtlas no debe inferirla por proximidad en Pokédex.
- **Learnset:** conocer los movimientos y conocer un Pokémon no implica saber qué movimientos puede aprender ese Pokémon.


In [19]:
demo_7 = preguntar_pokeatlas(
    "¿Cuál es la evolución de Pidgeotto?",
    thread_id="demo-evolucion",
)



👤 Usuario: ¿Cuál es la evolución de Pidgeotto?

🤖 PokéAtlas: No dispongo de esa información en mi base de conocimiento.

📚 Fragmentos recuperados:
  1. pokemon | Weedle | Kakuna | Beedrill | Pidgey | Pidgeotto | Pidgeot | pokemon_complete_clean.csv
  2. pokemon | Squirtle | Wartortle | Blastoise | Caterpie | Metapod | Butterfree | pokemon_complete_clean.csv
  3. pokemon | Steelix-Mega | Pidgeot-Mega | Glalie-Mega | Diancie-Mega | Metagross-Mega | Kyogre-Primal | pokemon_complete_clean.csv
  4. pokemon | Rattata | Raticate | Spearow | Fearow | Ekans | Arbok | pokemon_complete_clean.csv
  5. pokemon | Pupitar | Tyranitar | Lugia | Ho-Oh | Celebi | Treecko | pokemon_complete_clean.csv

🔎 Consulta usada para retrieval: ¿Cuál es la evolución de Pidgeotto?


In [20]:
demo_8 = preguntar_pokeatlas(
    "¿Cuál es el movimiento más fuerte que puede aprender Mewtwo?",
    thread_id="demo-learnset",
)



👤 Usuario: ¿Cuál es el movimiento más fuerte que puede aprender Mewtwo?

🤖 PokéAtlas: No dispongo de esa información en mi base de conocimiento.

📚 Fragmentos recuperados:
  1. move | Fotodestrucción Apocalíptica | Embestida Solar | moves.csv
  2. move | Disruptor Psíquico | Crioaliento Despiadado | moves.csv
  3. move | Deflagración Lunar | Somanta Amistosa | moves.csv
  4. move | Sísmico | Fuerza | moves.csv
  5. move | Gigarrayo Fulminante | Cabeza Sorpresa | moves.csv

🔎 Consulta usada para retrieval: ¿Cuál es el movimiento más fuerte que puede aprender Mewtwo?


## 12. Chat interactivo desde el notebook

Esta celda cumple el requisito de interacción textual.

**Importante:** mientras la celda esté ejecutándose, PokéAtlas estará esperando una pregunta en el cuadro de entrada. Escribe `salir` para terminar la conversación y finalizar la celda.


In [21]:
CHAT_THREAD_ID = "pokeatlas-chat-interactivo"

print("💬 Chat interactivo iniciado.")
print("La celda permanecerá ejecutándose mientras espera tus preguntas.")
print("Escribe 'salir' para terminar.\n")

while True:
    pregunta = input("Tú: ").strip()

    if pregunta.lower() in {"salir", "exit", "quit"}:
        print("PokéAtlas: ¡Hasta la próxima!")
        break

    if not pregunta:
        continue

    preguntar_pokeatlas(
        pregunta,
        thread_id=CHAT_THREAD_ID,
        mostrar_fuentes=False,
    )

    print("\n" + "=" * 80 + "\n")


💬 Chat interactivo iniciado.
La celda permanecerá ejecutándose mientras espera tus preguntas.
Escribe 'salir' para terminar.

PokéAtlas: ¡Hasta la próxima!


## 13. Qué demuestra este notebook

Con este notebook, PokéAtlas dispone de:

- **RAG** sobre una base vectorial propia en ChromaDB;
- **Gemini** como LLM y como proveedor de embeddings;
- **LangGraph** como orquestador del flujo;
- **System Prompt** específico y documentado;
- **memoria conversacional** mediante checkpointer y `thread_id`;
- resolución de referencias en preguntas de seguimiento;
- control explícito de respuestas cuando falta información;
- interacción textual desde Jupyter;
- más de cinco ejemplos de preguntas y respuestas para la demo.

### Limitación deliberada

`InMemorySaver` conserva la conversación mientras el proceso/kernel está vivo.
Para el alcance académico del notebook es suficiente como memoria conversacional
de sesión. Una aplicación desplegada podría sustituirlo por un checkpointer
persistente.


## 14. Decisiones finales de robustez

La versión final incorpora tres decisiones adicionales:

1. `gemini-3.5-flash-lite` como LLM para reducir coste/cuota en un chatbot.
2. El nodo de contextualización solo llama al LLM cuando detecta referencias al historial.
3. Los errores de cuota (`429`) se capturan y se convierten en mensajes comprensibles en lugar de mostrar un traceback.

Además, el System Prompt prohíbe inferir relaciones evolutivas o learnsets cuando esas relaciones no aparecen explícitamente en la base documental.
